# Clinopyroxene-Liquid Thermobarometry — Cumbre Vieja (La Palma)

**Caracciolo et al. — Thermobarometry code** · version `_Alberto_30.06.2026`

This notebook estimates the **pressure (P)** and **temperature (T)** of clinopyroxene crystallisation for the Cumbre Vieja eruptions using **Cpx-Liquid melt matching** with the [Thermobar](https://thermobar.readthedocs.io/) package (Wieser et al., 2022, https://doi.org/10.30909/vol.05.02.349384).

For every clinopyroxene (Cpx) and every candidate liquid (melt) composition, Thermobar forms all possible Cpx-Liquid pairs, keeps only pairs that pass equilibrium tests (Kd Fe-Mg, Ca-Ti, etc.), iteratively solves the chosen P and T equations to convergence, and averages the surviving matches for each clinopyroxene. Several **P-T calibrations** are run so the results can be compared between models. For each calibration, the averaged results are filtered with a Ca-Ti equilibrium criterion, converted to degrees Celsius, padded back to one row per input crystal, and saved as a spreadsheet.

The notebook is self-contained: it only needs **Thermobar** (plus numpy/pandas) and the per-eruption input spreadsheets, and it processes **all three eruptions** (El Charco 1712, Teneguía 1971, Tajogaite 2021) in one run. See the accompanying README for installation.

The **Masotta et al. (2013)** calibration (`PT_Masotta2013`) is applied only to the **evolved clinopyroxene cores** (Mg# < 67) and uses a different input melt from the other five models: a single **titanite-hosted, tephri-phonolitic melt-pocket** composition found in a plagioclase antecryst from the 2021 Tajogaite eruption (`Jegal2025_MI`; Jegal et al., 2025). To increase the number of matches, this single composition is expanded into a cloud of **1000** Monte Carlo melts generated in the code — each oxide drawn from a Gaussian with a **5 % (1σ) relative** spread — and run with **`H2O_Liq = 3` wt.%** and **`Fe3Fet_Liq = 0`** (Masotta assumes all Fe as Fe²⁺). These settings are controlled by the `Jegal2025_MI` and `MASOTTA_*` constants in the Configuration cell. 

## Install Thermobar (first time only)

In [ ]:
# Thermobar must be installed. If you have not installed it
# yet, uncomment the line below (remove the #) and run this cell.
#!pip install Thermobar

## Imports

In [ ]:
import numpy as np
import pandas as pd
import Thermobar as pt

print("Thermobar version:", pt.__version__)

## Configuration

Everything you might want to change is in this cell and the next: which eruptions to process, which P-T calibrations to run, the equilibrium filters, and the output folder.

In [ ]:
# Eruptions to process, each with its input spreadsheet (same folder).
ERUPTIONS = [
    {"name": "ElCharco1712",  "input_file": "Input_ElCharco1712.xlsx"},
    {"name": "Teneguia1971",  "input_file": "Input_Teneguia1971.xlsx"},
    {"name": "Tajogaite2021", "input_file": "Input_Tajogaite2021.xlsx"},
]

# ----------------------------------------------------------------------
# Pressure (P) + Temperature (T) calibrations to run.
# Each tuple is (equationP, equationT, label-used-in-output-filenames):
#   P_Mollo2018_AMAM   — Mollo et al. (2018) barometer
#   T_Put2008_eq33     — Putirka (2008) eq. 33 thermometer (H2O-dependent)
#   T_Mollo2018_eq33MAM— Mollo et al. (2018) thermometer
#   P_Put2003/T_Put2003— Putirka (2003) P and T
#   P_Neave2017        — Neave & Putirka (2017) barometer
# Comment/uncomment lines to change which models are run.
# ----------------------------------------------------------------------
MODEL_SETTINGS = [
    ("P_Mollo2018_AMAM", "T_Put2008_eq33",     "P_Mollo2018_Teq33"),
    ("P_Mollo2018_AMAM", "T_Mollo2018_eq33MAM", "PT_Mollo2018"),
    ("P_Put2003",        "T_Put2003",           "PT_P2003"),
    ("P_Put2003",        "T_Put2008_eq33",      "P_Put2003_Teq33"),
    ("P_Neave2017",      "T_Put2008_eq33",      "P_NP2017_T_Eq33"),
    ("P_Mas2013_Palk2012", "T_Mas2013_Talk2012", "PT_Masotta2013"),
]

# Parameters passed to the iterative Cpx-Liquid melt-matching routine.
# Kd_Err and the *_Err values set the equilibrium range; H2O_Liq
MATCHING_PARAMS = dict(
    Kd_Err=0.08, H2O_Liq=1, DiHd_Err=0.10, CaTs_Err=0.06,
    EnFs_Err=0.05, Fe3Fet_Liq=0.11, iterations=30,
)

# ------------------------------------------------------------------------------------------
# Masotta et al. (2013) — Model used for estimated P-T of evolved clinopyroxene compositions.
# ------------------------------------------------------------------------------------------
# 1) Masotta uses its owm input melt for ALL eruptions: a Monte Carlo cloud
#    of N melts generated in-code (no external file) around the Jegal2025_MI
#    composition below, perturbing each oxide by a 5% (1-sigma) relative
#    Gaussian. See generate_masotta_melts() in the next cell.
# 2) Masotta (2013) is calibrated assuming all Fe is Fe2+, so Fe3Fet_Liq=0.
# 3) H2O_Liq=3 wt.% (the other models keep H2O_Liq=1).
# 4) To get more pairs, the Ca-Ti equilibrium filter is skipped for Masotta.
# 5) Only P-T for evolved cpx (Mg# < 67) are retained for the Masotta model.
# These overrides are wired into run_models() / postprocess() below.
MASOTTA_LABEL       = "PT_Masotta2013"

# Titanite-hosted, tephri-phonolitic melt-pocket composition (oxide wt.%)
# in a plagioclase antecryst, 2021 Tajogaite eruption (Jegal et al., 2025).
Jegal2025_MI = {
    "SiO2_Liq": 50.9, "TiO2_Liq": 1.8, "Al2O3_Liq": 16.4,
    "FeOt_Liq": 10.1, "MnO_Liq": 0.2,  "MgO_Liq": 2.0,
    "CaO_Liq": 5.0,   "Na2O_Liq": 5.7, "K2O_Liq": 4.3, "P2O5_Liq": 1.1,
}
MASOTTA_N_SIM        = 1000      # number of Monte Carlo melts to generate
MASOTTA_REL_SIGMA    = 0.05      # 1-sigma = 5% of each oxide value
MASOTTA_MC_SEED      = 42        # reproducible draws (set None for random)
MASOTTA_FE3FET_LIQ   = 0         # all Fe as Fe2+ for Masotta consistency
MASOTTA_H2O_LIQ      = 3         # melt H2O (wt.%) for Masotta only
MASOTTA_SKIP_CATI    = True      # skip the Ca-Ti filter for Masotta
MASOTTA_CPX_MGNO_MAX = 67        # keep only Cpx with Mg# < this value

# Only Cpx-Liq pairs with a mean Ca-Ti exchange residual (Put1999) at or below this value are kept as equilibrium matches.
DELTA_CATI_MAX = 0.02

# Folder where the per-calibration result spreadsheets are written.
OUTPUT_DIR = "."

## Helper functions

The full workflow for one eruption is wrapped into four small functions: `load_inputs`, `run_models`, `postprocess` and `save_results`.

Each input spreadsheet has two sheets, **Melts** and **Cpxs**, holding the liquid and clinopyroxene major-element compositions (oxide wt.% columns such as `SiO2_Liq`,`MgO_Cpx`, …) plus a sample identifier column. They are read with Thermobar's `import_excel`, which returns the cleaned composition tables.

In [ ]:
def load_inputs(input_file):
    """Read the Melts (liquids) and Cpxs sheets of one eruption's input file.

    The Thermobar importer returns several frames; we keep the cleaned
    liquid and clinopyroxene composition tables used by the matching.
    """
    liqs = pt.import_excel(input_file, sheet_name="Melts")["Liqs"]
    cpxs = pt.import_excel(input_file, sheet_name="Cpxs")["Cpxs"]
    return liqs, cpxs


def generate_masotta_melts(mean_comp=Jegal2025_MI,
                           n_sim=MASOTTA_N_SIM,
                           rel_sigma=MASOTTA_REL_SIGMA,
                           seed=MASOTTA_MC_SEED):
    """Monte Carlo melt cloud for the Masotta (2013) calibration (no file).

    Draws ``n_sim`` synthetic melts: each oxide is sampled from a normal
    distribution centred on its mean value with a 1-sigma of ``rel_sigma``
    times that value (5% by default). Negative draws are clipped to 0.
    Returns a DataFrame in Thermobar *_Liq format, used as the input
    liquids for the Masotta calibration across all eruptions.
    """
    rng = np.random.default_rng(seed)
    data = {oxide: np.clip(rng.normal(mu, rel_sigma * mu, n_sim), 0, None)
            for oxide, mu in mean_comp.items()}
    melts = pd.DataFrame(data)
    melts.insert(0, "Sample_ID_Liq", [f"MC_{i + 1}" for i in range(n_sim)])
    return melts


def cpx_mgno(cpx_comps):
    """Clinopyroxene Mg# = 100 * molar Mg / (Mg + Fe_total), Fe as FeO."""
    n_mg = cpx_comps["MgO_Cpx"] / 40.3044   # molar mass MgO
    n_fe = cpx_comps["FeOt_Cpx"] / 71.8444  # molar mass FeO
    return 100.0 * n_mg / (n_mg + n_fe)


def run_models(liqs, cpxs, masotta_liqs=None, model_settings=MODEL_SETTINGS,
               matching_params=MATCHING_PARAMS):
    """Run the Cpx-Liquid melt matching for every (P, T) calibration.

    Returns a dict {label: averaged-PT DataFrame}. For each calibration the
    routine considers every Cpx-Liquid pair, filters to equilibrium pairs,
    iterates P-T to convergence, and averages the matches per clinopyroxene.

    The Masotta (2013) calibration (MASOTTA_LABEL) is special-cased:
      * input liquids are the Monte Carlo melts (``masotta_liqs``);
      * Fe3Fet_Liq is forced to 0 (all Fe as Fe2+);
      * H2O_Liq is set to MASOTTA_H2O_LIQ;
      * only averaged Cpx with Mg# < MASOTTA_CPX_MGNO_MAX are kept.
    """
    # Cpx indices to keep for the Masotta calibration (Mg# below threshold).
    masotta_keep_ids = set(cpxs.index[cpx_mgno(cpxs) < MASOTTA_CPX_MGNO_MAX])

    results = {}
    for eqP, eqT, label in model_settings:
        print(f"  Running model: {label}")

        model_liqs = liqs
        params = dict(matching_params)

        if label == MASOTTA_LABEL:
            if masotta_liqs is not None:
                model_liqs = masotta_liqs
                print(f"    Using {masotta_liqs.shape[0]} Monte Carlo melts.")
            params["Fe3Fet_Liq"] = MASOTTA_FE3FET_LIQ   # all Fe as Fe2+
            params["H2O_Liq"] = MASOTTA_H2O_LIQ         # melt H2O for Masotta

        res = pt.calculate_cpx_liq_press_temp_matching(
            liq_comps=model_liqs, cpx_comps=cpxs,
            equationP=eqP, equationT=eqT, **params,
        )
        av = res["Av_PTs"]

        if label == MASOTTA_LABEL and "ID_CPX" in av.columns:
            before = len(av)
            av = av[av["ID_CPX"].isin(masotta_keep_ids)].copy()
            print(f"    Cpx Mg# < {MASOTTA_CPX_MGNO_MAX} filter: "
                  f"{len(av)} of {before} averaged Cpx retained.")

        results[label] = av
    return results


def postprocess(av_df, label=None, delta_cati_max=DELTA_CATI_MAX):
    """Filter, convert and pad one averaged-PT table.

    Steps:
      1. keep only equilibrium matches (mean Ca-Ti residual <= threshold);
         this Ca-Ti filter is SKIPPED for the Masotta calibration
         (MASOTTA_LABEL) when MASOTTA_SKIP_CATI is True;
      2. add temperature in degrees Celsius (T_°C = T_K - 273.15);
      3. re-insert empty rows for clinopyroxenes that found no match, so the
         output keeps one row per input crystal (ID_CPX = 0..max);
      4. move the key P-T columns to the front.
    """
    skip_cati = (label == MASOTTA_LABEL) and MASOTTA_SKIP_CATI
    if skip_cati:
        print("    Skipping Ca-Ti filter for Masotta calibration.")
        df = av_df.copy()
    else:
        df = av_df[av_df["Mean_Delta_CaTi_Put1999"] <= delta_cati_max].copy()

    # Temperature in degrees Celsius
    df["T_°C"] = df["Mean_T_K_calc"] - 273.15

    # Re-insert empty rows for unmatched clinopyroxenes (one row per ID_CPX, from 0 to the largest input index) so the output keeps one row per crystal
    if "ID_CPX" in df.columns and df["ID_CPX"].notna().any():
        try:
            id_vals = df["ID_CPX"].dropna().astype(int)
            full_ids = range(0, id_vals.max() + 1)
            df = (df.assign(ID_CPX=df["ID_CPX"].astype("Int64"))
                    .set_index("ID_CPX")
                    .reindex(full_ids)
                    .rename_axis("ID_CPX")
                    .reset_index())
        except (ValueError, TypeError):
            print("    Could not pad missing ID_CPX rows; leaving as is.")

    # Bring the most useful columns to the front
    key_cols = ["Mean_P_kbar_calc", "Std_P_kbar_calc", "T_°C",
                "Std_T_K_calc", "# of Liqs Averaged", "Mean_T_K_calc"]
    front = [c for c in key_cols if c in df.columns]
    rest = [c for c in df.columns if c not in front]
    return df[front + rest]


def save_results(filtered_results, eruption_name, output_dir=OUTPUT_DIR):
    """Write one spreadsheet per calibration: <label>_<eruption>_Output.xlsx."""
    for label, df in filtered_results.items():
        path = f"{output_dir}/{label}_{eruption_name}_Output.xlsx"
        df.to_excel(path, index=False)
        print(f"  Saved: {path}")

## Run all eruptions

Loops over every eruption: loads its inputs, runs all calibrations, post-processes and saves the results. The averaged tables are also kept in `all_results` for any further inspection in the notebook.

**Note:** the Tajogaite 2021 dataset is large (~1.3 million Cpx-Liquid pairs per calibration), so a full run takes several minutes.

In [ ]:
all_results = {}

# Melts used ONLY by the Masotta (2013) calibration, shared across all
# eruptions: a 1000-simulation Monte Carlo cloud generated in-code (5% 1-sigma)
# around the Jegal2025_MI melt. No external melt file is read.
masotta_liqs = generate_masotta_melts()
print(f"Generated {masotta_liqs.shape[0]} Monte Carlo melts for Masotta "
      f"({int(MASOTTA_REL_SIGMA * 100)}% 1-sigma).")

for eruption in ERUPTIONS:
    print(f"\n=== {eruption['name']} ===")
    liqs, cpxs = load_inputs(eruption["input_file"])
    print(f"Loaded {liqs.shape[0]} liquids and {cpxs.shape[0]} clinopyroxenes.")

    raw_results = run_models(liqs, cpxs, masotta_liqs=masotta_liqs)
    filtered_results = {label: postprocess(av_df, label=label)
                        for label, av_df in raw_results.items()}
    save_results(filtered_results, eruption["name"])

    all_results[eruption["name"]] = filtered_results

print("\nAll eruptions processed.")